In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta import *

In [2]:
# 1. Configuração da Sessão Spark
warehouse_location = 'hdfs://hdfs-nn:9000/Projeto/Silver'

builder = SparkSession.builder \
    .appName("Gold - Fact Performance Audiovisual") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
# 2. Carregar Tabelas
spark.read.format("delta").load("hdfs://hdfs-nn:9000/projeto/silver/streaming_titles").createOrReplaceTempView("silver_streaming")
spark.read.format("delta").load("hdfs://hdfs-nn:9000/projeto/silver/rotten_tomatoes").createOrReplaceTempView("silver_rotten")
spark.read.format("delta").load("hdfs://hdfs-nn:9000/projeto/silver/box_office").createOrReplaceTempView("silver_box_office")
spark.read.format("delta").load("hdfs://hdfs-nn:9000/gold/gold_projeto.db/dim_tempo").createOrReplaceTempView("dim_tempo")
spark.read.format("delta").load("hdfs://hdfs-nn:9000/gold/gold_projeto.db/dim_conteudo").createOrReplaceTempView("dim_conteudo")

In [4]:
# 3. Consolidação (Removendo duplicados e coluna de plataforma)
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_streaming_consolidated AS
    SELECT DISTINCT 
        title, 
        release_year, 
        imdb_score, 
        tmdb_score, 
        imdb_votes, 
        tmdb_popularity 
    FROM silver_streaming
""").collect()

[]

In [5]:
# 4. Processamento da Fact Table
# Cruzamento com Box Office (Profit) e Rotten Tomatoes (Status)
df_fact_base = spark.sql("""
    WITH dim_tempo_yearly AS (
        SELECT year, FIRST(id_data) as id_time 
        FROM dim_tempo 
        GROUP BY year
    )
    SELECT 
        -- Renomeação para chaves forasteiras (FK)
        c.id_conteudo as fk_content,
        t.id_time as fk_time,
        CAST(b.Gross AS DOUBLE) as audiovisual_profit, 
        CAST(r.tomatometer_rating AS INT) as critic_score,  
        CAST(r.audience_rating AS INT) as public_score,  
        CAST(s.tmdb_popularity AS DOUBLE) as popularity_index, 
        CAST(s.imdb_votes AS INT) as total_votes,   
        -- Status do Rotten Tomatoes (Ex: Certified Fresh, Rotten). Se nulo, fica 'N/A'
        COALESCE(r.tomatometer_status, 'N/A') as rotten_status
    FROM v_streaming_consolidated s
    -- Join com Box Office (Matches exatos de Título e Ano)
    INNER JOIN silver_box_office b ON 
        LOWER(TRIM(s.title)) = LOWER(TRIM(b.Title)) AND s.release_year = b.Year
    -- Join com Rotten Tomatoes (Left Join, pois nem todos têm status)
    LEFT JOIN silver_rotten r ON
        LOWER(TRIM(s.title)) = LOWER(TRIM(r.movie_title))
    -- Join com Dimensão Tempo
    LEFT JOIN dim_tempo_yearly t ON 
        s.release_year = t.year
    -- Join com Dimensão Conteúdo (Para obter o ID da dimensão)
    LEFT JOIN dim_conteudo c ON
        LOWER(TRIM(s.title)) = LOWER(TRIM(c.title))
    WHERE c.id_conteudo IS NOT NULL
""")

In [6]:
# 5. Adicionar Primary Key (id_performance) sequencial
windowSpec = Window.orderBy("fk_content")
df_fact_final = df_fact_base.withColumn("id_performance", F.row_number().over(windowSpec))

# Reordenar colunas com os novos nomes
df_fact_final = df_fact_final.select(
    "id_performance", "fk_content", "fk_time", 
    "audiovisual_profit", "critic_score", "public_score", 
    "popularity_index", "total_votes", "rotten_status"
)

In [7]:
df_fact_final.printSchema()
df_fact_final.show()

root
 |-- id_performance: integer (nullable = false)
 |-- fk_content: integer (nullable = true)
 |-- fk_time: integer (nullable = true)
 |-- audiovisual_profit: double (nullable = true)
 |-- critic_score: integer (nullable = true)
 |-- public_score: integer (nullable = true)
 |-- popularity_index: double (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- rotten_status: string (nullable = false)

+--------------+----------+-------+------------------+------------+------------+----------------+-----------+---------------+
|id_performance|fk_content|fk_time|audiovisual_profit|critic_score|public_score|popularity_index|total_votes|  rotten_status|
+--------------+----------+-------+------------------+------------+------------+----------------+-----------+---------------+
|             1|         2|   7080|       7.8132231E7|        null|        null|           6.135|        117|            N/A|
|             2|         5|   5283|       6.9055695E7|          29|          62| 

In [8]:
# 6. Escrita na Gold
target_path = "hdfs://hdfs-nn:9000/gold/gold_projeto.db/fact_performance_audiovisual"

# Limpar tabela anterior
spark.sql("DROP TABLE IF EXISTS gold_projeto.fact_performance_audiovisual")

df_fact_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", target_path) \
    .saveAsTable("gold_projeto.fact_performance_audiovisual")

In [10]:
from pyspark.sql import functions as F

# 1. Carregar a tabela final para validação
df_check = spark.table("gold_projeto.fact_performance_audiovisual")

print("="*50)
print("ESTRUTURA E VOLUMETRIA")
print("="*50)

# Verificação do Schema
print("\n1. Estrutura da Tabela (Schema):")
df_check.printSchema()

# Contagem Total
total_registos = df_check.count()
print(f"2. Número total de registos: {total_registos}")

print("\n" + "="*50)
print("AMOSTRA DE DADOS (TOP 10)")
print("="*50)

# Mostrar as primeiras linhas para verificar o preenchimento do rotten_status
df_check.orderBy("id_performance").show(10, truncate=False)

print("\n" + "="*50)
print("DISTRIBUIÇÃO DE STATUS ROTTEN TOMATOES")
print("="*50)

df_check.groupBy("rotten_status").count().orderBy(F.col("count").desc()).show()

print("="*50)
print("VERIFICAÇÃO CONCLUÍDA")
print("="*50)

ESTRUTURA E VOLUMETRIA

1. Estrutura da Tabela (Schema):
root
 |-- id_performance: integer (nullable = true)
 |-- fk_content: integer (nullable = true)
 |-- fk_time: integer (nullable = true)
 |-- audiovisual_profit: double (nullable = true)
 |-- critic_score: integer (nullable = true)
 |-- public_score: integer (nullable = true)
 |-- popularity_index: double (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- rotten_status: string (nullable = true)

2. Número total de registos: 824

AMOSTRA DE DADOS (TOP 10)
+--------------+----------+-------+------------------+------------+------------+----------------+-----------+-------------+
|id_performance|fk_content|fk_time|audiovisual_profit|critic_score|public_score|popularity_index|total_votes|rotten_status|
+--------------+----------+-------+------------------+------------+------------+----------------+-----------+-------------+
|1             |2         |7080   |7.8132231E7       |null        |null        |6.135           |1

In [11]:
from pyspark.sql import functions as F

# 1. Carregar as tabelas da Gold
df_facto = spark.table("gold_projeto.fact_performance_audiovisual")
df_conteudo = spark.table("gold_projeto.dim_conteudo")

print("="*60)
print("ESTRUTURA E VOLUMETRIA")
print("="*60)

# Contagem Total e Schema
print(f"1. Número total de registos na Facto: {df_facto.count()}")
df_facto.printSchema()

print("\n" + "="*60)
print("AMOSTRA DE DADOS (TOP 15 LINHAS)")
print("="*60)
# Mostrar as primeiras 15 linhas ordenadas pela nova PK
df_facto.orderBy("id_performance").show(15, truncate=False)

print("\n" + "="*60)
print("EXPLORAÇÃO DE DADOS (SQL QUERIES)")
print("="*60)

# Criar vistas temporárias para SQL
df_facto.createOrReplaceTempView("facto_teste")
df_conteudo.createOrReplaceTempView("dim_conteudo_teste")

# Query 1: Volume de lançamentos por ano
print("\n1. Quantidade de filmes por ano de lançamento (na Facto):")
spark.sql("""
    SELECT release_year, COUNT(*) as total_titles
    FROM dim_conteudo_teste
    WHERE id_conteudo IN (SELECT fk_content FROM facto_teste)
    GROUP BY release_year
    ORDER BY release_year DESC
    LIMIT 10
""").show()

# Query 2: Filmes com maior discrepância (Público vs Crítica)
print("\n2. Filmes onde o público gostou muito mais que a crítica (Top 5):")
spark.sql("""
    SELECT c.title, f.public_score, f.critic_score, 
           ROUND((f.public_score - f.critic_score), 2) as diff,
           f.rotten_status
    FROM facto_teste f
    JOIN dim_conteudo_teste c ON f.fk_content = c.id_conteudo
    WHERE f.critic_score > 0
    ORDER BY diff DESC
    LIMIT 5
""").show()

# Query 3: Pesquisa por palavra-chave (Ex: 'Dark')
print("\n3. Verificação de métricas para títulos contendo 'Dark':")
spark.sql("""
    SELECT c.title, f.audiovisual_profit, f.popularity_index, f.rotten_status
    FROM facto_teste f
    JOIN dim_conteudo_teste c ON f.fk_content = c.id_conteudo
    WHERE LOWER(c.title) LIKE '%dark%'
""").show()

# Query 4: Estatística de Lucratividade
print("\n4. % de filmes com lucro acima de 100 Milhões:")
spark.sql("""
    SELECT 
        COUNT(*) as total_titles,
        SUM(CASE WHEN audiovisual_profit > 100000000 THEN 1 ELSE 0 END) as high_profit_titles,
        ROUND((SUM(CASE WHEN audiovisual_profit > 100000000 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)), 2) as percentage
    FROM facto_teste
""").show()

# Query 5: Análise do Status Rotten Tomatoes
print("\n5. Distribuição de Filmes por Status Rotten Tomatoes:")
spark.sql("""
    SELECT 
        rotten_status, 
        COUNT(*) as total,
        ROUND(AVG(audiovisual_profit), 0) as avg_profit,
        ROUND(AVG(public_score), 1) as avg_public_score
    FROM facto_teste
    GROUP BY rotten_status
    ORDER BY total DESC
""").show()

print("="*60)
print("VERIFICAÇÕES CONCLUÍDAS")
print("="*60)

ESTRUTURA E VOLUMETRIA
1. Número total de registos na Facto: 824
root
 |-- id_performance: integer (nullable = true)
 |-- fk_content: integer (nullable = true)
 |-- fk_time: integer (nullable = true)
 |-- audiovisual_profit: double (nullable = true)
 |-- critic_score: integer (nullable = true)
 |-- public_score: integer (nullable = true)
 |-- popularity_index: double (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- rotten_status: string (nullable = true)


AMOSTRA DE DADOS (TOP 15 LINHAS)
+--------------+----------+-------+------------------+------------+------------+----------------+-----------+---------------+
|id_performance|fk_content|fk_time|audiovisual_profit|critic_score|public_score|popularity_index|total_votes|rotten_status  |
+--------------+----------+-------+------------------+------------+------------+----------------+-----------+---------------+
|1             |2         |7080   |7.8132231E7       |null        |null        |6.135           |117        |N

In [14]:
# Query para investigar o género Reality
df_reality_investigacao = spark.sql("""
    SELECT 
        c.title,
        c.release_year,
        f.critic_score,
        f.public_score,
        (f.public_score - f.critic_score) as gap_diferenca,
        f.id_performance
    FROM gold_projeto.fact_performance_audiovisual f
    JOIN gold_projeto.dim_conteudo c ON f.fk_content = c.id_conteudo
    WHERE 
        c.genre_reality = 1                -- Filtrar apenas o género Reality
        AND f.critic_score IS NOT NULL     -- Garantir que temos as duas notas
        AND f.public_score IS NOT NULL
    ORDER BY 
        (f.public_score - f.critic_score) ASC -- Ordenar pelos mais negativos (Crítica > Público)
""")

print("--- Detalhe dos Títulos Reality com Maior Discrepância ---")
df_reality_investigacao.show(20, truncate=False)

# Contagem para ver se a amostra é pequena (o que explicaria a distorção)
print("--- Estatísticas Gerais de Reality ---")
spark.sql("""
    SELECT 
        COUNT(*) as total_titulos_reality,
        AVG(critic_score) as media_critica,
        AVG(public_score) as media_publico
    FROM gold_projeto.fact_performance_audiovisual f
    JOIN gold_projeto.dim_conteudo c ON f.fk_content = c.id_conteudo
    WHERE c.genre_reality = 1 
    AND f.critic_score IS NOT NULL
""").show()

--- Detalhe dos Títulos Reality com Maior Discrepância ---
+----------+------------+------------+------------+-------------+--------------+
|title     |release_year|critic_score|public_score|gap_diferenca|id_performance|
+----------+------------+------------+------------+-------------+--------------+
|date night|2019        |66          |55          |-11          |802           |
|fearless  |2016        |87          |79          |-8           |809           |
|blown away|2019        |32          |40          |8            |793           |
+----------+------------+------------+------------+-------------+--------------+

--- Estatísticas Gerais de Reality ---
+---------------------+------------------+-------------+
|total_titulos_reality|     media_critica|media_publico|
+---------------------+------------------+-------------+
|                    3|61.666666666666664|         58.0|
+---------------------+------------------+-------------+



In [ ]:
spark.stop()